# Features and Fold Split — CYP450 Direct Inhibition

Produces two frozen artifacts that every subsequent notebook loads and never
regenerates: the CV fold assignments, and the feature sets for the tabular baselines
(ECFP4 + physicochemical descriptors, and CheMeleon foundation-model embeddings). Also
lays out the multitask target table.

**This notebook does:**
- loads the curated train/test files built in `01_data_curation.ipynb` and re-confirms
  row counts/per-isoform label counts against the figures already verified there
- generates a plain 5×5 repeated random CV fold assignment over the training compounds
  (Ash et al.), checks whether CYP2D6 needs separate stratification consideration, and
  freezes the result to disk
- builds ECFP4 + physicochemical-descriptor tabular features via the existing functions
  in `src/features.py`, for every compound in the curated train and test sets
- computes CheMeleon foundation-model embeddings (a frozen forward pass, not a training
  run) for the same compound set
- lays out the four isoform pIC50 columns for multitask use, with explicit NaN handling
- reconciles the single-concentration primary-screen compound count against the pIC50
  training set, a number 03b's design needs

**This notebook does NOT do:** any chemistry-motivated / cluster-based split (that's a
separate, still-open core-tier decision, out of scope here), any model training, or any
03b log2fc-pretraining logic — it only reports the compound-count numbers 03b needs to
make that decision.

Per `CLAUDE.md`: row counts are logged before/after every step; every seed used is
logged; fingerprint/descriptor/embedding logic lives in `src/features.py`, not
duplicated inline; every modelling-adjacent decision not explicitly settled by the task
brief is flagged and reasoned through inline, not silently picked.

In [1]:
# OMP_NUM_THREADS must be set before rdkit/torch are first imported anywhere in this
# process -- rdkit and torch each bundle their own OpenMP runtime on macOS, and running
# them multi-threaded in the same process segfaults (confirmed directly during
# development of `chemeleon_embeddings` in src/features.py -- see Section 3). Single
# thread is the only configuration found to be reliable; see Section 3 for the full
# story and the runtime cost this has for that one cell.
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedKFold
from rdkit import DataStructs

from src.features import (
    ecfp4_fingerprints,
    isoform_structural_descriptors,
    rdkit_2d_descriptors,
    chemeleon_embeddings,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

RAW = 'data/raw'          # relative to repo root -- run this notebook with cwd = repo root
PROCESSED = 'data/processed'
FOLDS = 'data/folds'
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
PIC50_COLS = {iso: f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS}

SEED = 42
print('master random seed for this notebook:', SEED)

master random seed for this notebook: 42


## 0. Load curated data and confirm row counts

Loads the two files built in `01_data_curation.ipynb` -- no raw files are touched here.
Per `CLAUDE.md`, row counts and per-isoform counts are re-confirmed against the figures
already verified in `00_schema_audit.ipynb` / `01_data_curation.ipynb` rather than
assumed.

In [2]:
train = pd.read_csv(f'{PROCESSED}/train_inhibition_curated.csv')
test = pd.read_csv(f'{PROCESSED}/test_blinded_curated.csv')

print('train_inhibition_curated.csv shape:', train.shape)
print('test_blinded_curated.csv shape:', test.shape)
print()
print('per-isoform non-null pIC50 counts (train):')
for iso in ISOFORMS:
    n = int(train[PIC50_COLS[iso]].notna().sum())
    print(f'  {iso}: {n}')

train_inhibition_curated.csv shape: (4905, 20)
test_blinded_curated.csv shape: (750, 4)

per-isoform non-null pIC50 counts (train):
  CYP1A2: 1412
  CYP2C9: 1285
  CYP2D6: 1493
  CYP3A4: 2335


Matches `01_data_curation.ipynb` exactly: 4,905 train rows / 750 test rows, and the
same per-isoform counts verified there against the schema audit (CYP1A2 1,412 / CYP2C9
1,285 / CYP2D6 1,493 / CYP3A4 2,335). Nothing filtered or dropped here.

## 1. CV fold assignments — plain 5×5 repeated random CV

Per the task brief: **plain random CV (Ash et al.), not a chemistry-motivated split.**
Cluster/scaffold-aware splitting is a separate, still-open core-tier decision and is
explicitly out of scope here — this section does not touch that question.

**Unit of splitting:** the 4,905 training compounds (by InChIKey), not individual
isoform-label rows. This is a single shared compound-level CV scheme reused across all
four isoform targets in the multitask setup — a compound held out in fold *k* of repeat
*r* is held out for every isoform it happens to have a label for, in that fold/repeat.
`01_data_curation.ipynb` already confirmed 0 duplicate InChIKeys within the training
set, so InChIKey and `Molecule_Name` are a 1:1 key here — no dedup step is needed before
splitting.

**Method:** `sklearn.model_selection.RepeatedKFold(n_splits=5, n_repeats=5,
random_state=SEED)` — 5 repeats of a 5-fold split, i.e. 5 repeat columns per compound
(`repeat_0`..`repeat_4`), each valued 0-4. "5 repeats × 5 folds = 25" refers to the 25
distinct (repeat, fold) identities in the overall design, not 25 columns per compound —
noted explicitly since the task brief's phrasing could be read either way.

**Seed handling, re: `CLAUDE.md`'s "seeds must vary per fold, not be shared globally"
rule:** this rule is about seeds used when *fitting models* per CV fold (relevant to
03b, which does the actual model training) — no model is fit in this notebook, only the
partition itself is generated. For the partition, one master seed (`SEED = 42`, logged
above) is passed to `RepeatedKFold`. Checked directly in `sklearn`'s source
(`_RepeatedSplits.split`): each repeat constructs a fresh `KFold(random_state=rng,
shuffle=True)` where `rng` is a single `RandomState` that keeps advancing across
repeats — so the 5 repeats are *not* 5 copies of the same partition seeded once, they
are 5 genuinely different shuffles drawn from one continuing stream. This is verified
empirically below (Section 1b), not just asserted.

In [3]:
rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=SEED)

n_train = len(train)
splits = list(rkf.split(np.arange(n_train)))
assert len(splits) == 25, f'expected 25 (repeat,fold) splits, got {len(splits)}'

fold_matrix = np.full((n_train, 5), -1, dtype=int)
for repeat_id in range(5):
    for fold_id in range(5):
        _, held_out_idx = splits[repeat_id * 5 + fold_id]
        fold_matrix[held_out_idx, repeat_id] = fold_id

assert (fold_matrix >= 0).all(), 'every compound must get a fold id in every repeat'
print('fold_matrix shape:', fold_matrix.shape, '(n_train_compounds x 5 repeats)')
print()
print('fold-size balance per repeat (should be ~981/981/981/981/981):')
for repeat_id in range(5):
    print(f'  repeat_{repeat_id}:', np.bincount(fold_matrix[:, repeat_id]).tolist())

fold_matrix shape: (4905, 5) (n_train_compounds x 5 repeats)

fold-size balance per repeat (should be ~981/981/981/981/981):
  repeat_0: [981, 981, 981, 981, 981]
  repeat_1: [981, 981, 981, 981, 981]
  repeat_2: [981, 981, 981, 981, 981]
  repeat_3: [981, 981, 981, 981, 981]
  repeat_4: [981, 981, 981, 981, 981]


### 1a. CYP2D6 stratification — checked, not assumed

The task brief asks whether CYP2D6 needs separate stratification consideration given
its "different assay modality" from the other three isoforms. Checked against what's
actually documented in the released files: `00_schema_audit.ipynb` found CYP2D6 uses
the identical `pIC50_direct_inhibition` / `_conf_high` / `_conf_low` / `_std` column
schema as the other three isoforms in `TRAIN_inhibition.csv` — no assay-protocol
documentation anywhere in the released files distinguishes CYP2D6's measurement method
from CYP1A2/CYP2C9/CYP3A4 for the Direct Inhibition endpoint. (The one place a CYP2D6
asymmetry *was* found — `CYP2D6_is_TDI`/`CYP3A4_is_TDI` existing in `TRAIN_TDI.csv`
without CYP1A2/CYP2C9 equivalents — was already flagged in the schema audit and
resolved there as TDI-track-only, irrelevant to Direct Inhibition.)

Since no verifiable protocol distinction exists to reason from, this checks the thing a
"different modality" concern would actually threaten in a *plain random* compound-level
CV design: does CYP2D6 end up under/over-represented in some folds by chance? Computed
directly below, for all four isoforms, across all 25 (repeat, fold) partitions.

In [4]:
print('per-isoform non-null count per fold, across all 25 (repeat,fold) partitions:')
print(f'{"isoform":<8} {"total":>6} {"expected/fold":>14} {"mean":>8} {"std":>7} {"min":>6} {"max":>6} {"rel.std":>8}')
for iso in ISOFORMS:
    non_null = train[PIC50_COLS[iso]].notna().to_numpy()
    total = int(non_null.sum())
    counts = np.array([
        non_null[fold_matrix[:, r] == f].sum()
        for r in range(5) for f in range(5)
    ])
    expected = total / 5
    rel_std = counts.std() / counts.mean() * 100
    print(f'{iso:<8} {total:>6} {expected:>14.1f} {counts.mean():>8.1f} {counts.std():>7.2f} '
          f'{counts.min():>6} {counts.max():>6} {rel_std:>7.1f}%')

per-isoform non-null count per fold, across all 25 (repeat,fold) partitions:
isoform   total  expected/fold     mean     std    min    max  rel.std
CYP1A2     1412          282.4    282.4   12.72    252    307     4.5%
CYP2C9     1285          257.0    257.0   13.32    227    278     5.2%
CYP2D6     1493          298.6    298.6   12.78    276    322     4.3%
CYP3A4     2335          467.0    467.0   15.93    440    509     3.4%


**Decision: no CYP2D6-specific stratification; all four isoforms treated uniformly for
CV purposes.** Per-fold CYP2D6 representation under plain random splitting is already
tightly balanced: 4.3% relative std, fold counts ranging 276-322 against an expected
298.6 (within ~8%) — essentially identical to the other three isoforms' balance (3.4-
5.2% relative std, see the table above), and consistent with simple hypergeometric
variance for a ~1,500-of-4,905 label drawn into 5 roughly-equal random folds. There is
no documented assay-protocol basis in the released files to treat CYP2D6 differently,
and the one thing that's actually checkable — fold-level label balance — shows no
problem to correct for. If a genuine protocol distinction surfaces later (e.g. from
OpenADMET directly, as happened for the `_conf_high`/`_conf_low` question in the schema
audit's follow-up), this decision should be revisited.

### 1b. Verify repeats are genuinely distinct (not 5 copies of one partition)

In [5]:
print('fraction of compounds with the same fold id between each pair of repeats:')
print('(≈0.20 expected under independent random 5-way assignment; ≈1.00 would mean a repeat is a duplicate)')
for i in range(5):
    for j in range(i + 1, 5):
        frac_same = (fold_matrix[:, i] == fold_matrix[:, j]).mean()
        print(f'  repeat_{i} vs repeat_{j}: {frac_same:.3f}')

fraction of compounds with the same fold id between each pair of repeats:
(≈0.20 expected under independent random 5-way assignment; ≈1.00 would mean a repeat is a duplicate)
  repeat_0 vs repeat_1: 0.196
  repeat_0 vs repeat_2: 0.192
  repeat_0 vs repeat_3: 0.205
  repeat_0 vs repeat_4: 0.199
  repeat_1 vs repeat_2: 0.199
  repeat_1 vs repeat_3: 0.201
  repeat_1 vs repeat_4: 0.192
  repeat_2 vs repeat_3: 0.209
  repeat_2 vs repeat_4: 0.203
  repeat_3 vs repeat_4: 0.198


All 10 pairs land close to 0.20, the value expected if two repeats' fold assignments
were independent — confirms the 5 repeats are genuinely different partitions, not the
same split repeated, and confirms empirically (not just from reading `sklearn`'s source)
that one master seed correctly produces 5 distinct shuffles here.

### 1c. Confirm alignment against the curated data, then save

Builds the output table (`Molecule_Name`, `inchikey`, `repeat_0`..`repeat_4`) directly
from `train`'s own row order and key columns, then re-loads it from disk and merges it
back against `train_inhibition_curated.csv` on `inchikey` as an explicit alignment
check — not just "correct by construction," actually verified, per the task brief's
instruction to stop on any mismatch here.

In [6]:
os.makedirs(FOLDS, exist_ok=True)

fold_cols = [f'repeat_{r}' for r in range(5)]
cv_folds = pd.DataFrame(fold_matrix, columns=fold_cols)
cv_folds.insert(0, 'inchikey', train['inchikey'].values)
cv_folds.insert(0, 'Molecule_Name', train['Molecule_Name'].values)

print('rows before save:', len(cv_folds))
folds_path = f'{FOLDS}/cv_folds.csv'
cv_folds.to_csv(folds_path, index=False)
print('wrote', folds_path)

reread_folds = pd.read_csv(folds_path)
print('rows after reading back from disk:', len(reread_folds))
assert len(reread_folds) == len(train), 'fold file row count does not match curated training set'

check = train[['Molecule_Name', 'inchikey']].merge(reread_folds, on=['Molecule_Name', 'inchikey'], how='outer', indicator=True)
mismatch = check[check['_merge'] != 'both']
print('compounds not present in BOTH curated data and fold file:', len(mismatch))
assert len(mismatch) == 0, 'alignment mismatch between fold file and curated training data -- stopping'
assert reread_folds[fold_cols].isna().sum().sum() == 0, 'fold file has missing fold assignments'
print('alignment confirmed: fold file covers exactly the 4,905 curated training compounds, no missing values.')

rows before save: 4905
wrote data/folds/cv_folds.csv
rows after reading back from disk: 4905
compounds not present in BOTH curated data and fold file: 0
alignment confirmed: fold file covers exactly the 4,905 curated training compounds, no missing values.


## 2. Tabular baseline features — two descriptor options

Two feature files are built here, for two different purposes, both via existing
functions in `src/features.py` — no fingerprint/descriptor logic is reimplemented in
either: **(2a)** ECFP4 + a narrow, pharmacologically-motivated 9-descriptor set (already
used for notebook 02's isoform SAR interpretability work), and **(2b)** ECFP4's
descriptor companion widened to the full RDKit 2D descriptor set, added here to match
the feature recipe used by this challenge's own official baseline models. Neither
replaces the other; both are saved as separate files.

### 2a. ECFP4 + physicochemical descriptors (narrow, SAR-motivated set)

Computed via the existing `ecfp4_fingerprints` and `isoform_structural_descriptors`
functions in `src/features.py` — no fingerprint/descriptor logic is reimplemented here.

**On reusing `isoform_structural_descriptors` as "the" physicochemical descriptor
set:** this function was originally written in `02_chemical_space_exploration.ipynb`
for isoform-specific SAR checks (Kiani & Jabeen determinants: MW, HBA, HBD, ring count,
stereocenters, formal charge, logP, a logD proxy, a vsa_acc proxy). It is also the only
physicochemical-descriptor function that currently exists in `src/features.py`, so per
the task brief's "via the existing functions... do not reimplement" instruction, it is
reused here as-is for the tabular baseline rather than writing a second, overlapping
descriptor function. Its `logd_proxy`/`vsa_acc_proxy` caveats (documented in the
function's own docstring) carry over unchanged.

**Compound coverage:** every compound in the curated train **and** test sets (5,655
total, union) — not train-only. These are input features, not labels; computing them
for the blinded test set's SMILES doesn't touch any test-set label (there are none) and
is necessary for eventually generating predictions on it. A `split` column
(`train`/`test`) is carried in the output so downstream notebooks can filter.

In [7]:
compound_index = pd.concat([
    train[['Molecule_Name', 'inchikey', 'canonical_smiles']].assign(split='train'),
    test[['Molecule_Name', 'inchikey', 'canonical_smiles']].assign(split='test'),
], ignore_index=True)

print('rows before feature computation:', len(compound_index),
      f'(train {len(train)} + test {len(test)})')
assert compound_index['inchikey'].is_unique, 'expected no duplicate compounds across train+test union'

rows before feature computation: 5655 (train 4905 + test 750)


In [8]:
smiles_list = compound_index['canonical_smiles'].tolist()

fps = ecfp4_fingerprints(smiles_list)
n_fail = sum(fp is None for fp in fps)
print('ECFP4 parse failures:', n_fail, 'of', len(fps))
assert n_fail == 0, 'unexpected parse failure on already-canonicalized SMILES -- stopping'

ecfp4_arr = np.zeros((len(fps), 2048), dtype=np.uint8)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, ecfp4_arr[i])
ecfp4_cols = [f'ecfp4_{i:04d}' for i in range(2048)]
ecfp4_df = pd.DataFrame(ecfp4_arr, columns=ecfp4_cols)

desc_records = [isoform_structural_descriptors(s) for s in smiles_list]
desc_df = pd.DataFrame(desc_records)
n_desc_fail = int(desc_df.isna().any(axis=1).sum())
print('descriptor computation failures (any-NaN row):', n_desc_fail, 'of', len(desc_df))
assert n_desc_fail == 0, 'unexpected descriptor failure on already-canonicalized SMILES -- stopping'

tabular_features = pd.concat(
    [compound_index[['Molecule_Name', 'inchikey', 'split']].reset_index(drop=True), ecfp4_df, desc_df],
    axis=1,
)
print('tabular_features shape:', tabular_features.shape,
      f'({len(ecfp4_cols)} ECFP4 bits + {desc_df.shape[1]} descriptors + 3 id/split cols)')

ECFP4 parse failures: 0 of 5655


descriptor computation failures (any-NaN row): 0 of 5655
tabular_features shape: (5655, 2060) (2048 ECFP4 bits + 9 descriptors + 3 id/split cols)


In [9]:
os.makedirs(PROCESSED, exist_ok=True)
tabular_path = f'{PROCESSED}/tabular_baseline_features.csv'

print('rows before save:', len(tabular_features))
tabular_features.to_csv(tabular_path, index=False)
print('wrote', tabular_path)

reread_tabular = pd.read_csv(tabular_path)
print('rows after reading back from disk:', len(reread_tabular))
assert len(reread_tabular) == len(tabular_features), 'tabular feature file row count changed on round-trip'
assert reread_tabular['inchikey'].is_unique
print('split counts:', reread_tabular['split'].value_counts().to_dict())

rows before save: 5655


wrote data/processed/tabular_baseline_features.csv


rows after reading back from disk: 5655
split counts: {'train': 4905, 'test': 750}


### 2b. Full RDKit 2D descriptors (matches the official baseline recipe)

The official OpenADMET baseline models for this challenge (XGB-baseline, LGBM-baseline
— confirmed via the project's OpenADMET Discord thread) use the **full** RDKit 2D
descriptor set concatenated with ECFP4, not a hand-picked subset. `isoform_structural_
descriptors` (used in 2a above) is deliberately narrow and was built for notebook 02's
isoform SAR interpretability work — useful for that purpose, but narrower than what the
models being benchmarked against actually use. `rdkit_2d_descriptors` (new in
`src/features.py`, via `Descriptors.CalcMolDescriptors`) closes that gap as a second
feature option, computed here for the same 5,655-compound (train ∪ test) union as 2a,
and saved to its own file rather than merged into `tabular_baseline_features.csv`.

Column count is not hard-coded anywhere here or in `src/features.py` — it's whatever
the installed RDKit version's `CalcMolDescriptors` produces, logged below alongside the
RDKit version itself, so this artifact is traceable to the exact toolchain that built
it.

In [10]:
import rdkit
print('rdkit version (determines the rdkit_2d_descriptors column set):', rdkit.__version__)

rdkit2d_records = [rdkit_2d_descriptors(s) for s in smiles_list]
rdkit2d_df = pd.DataFrame(rdkit2d_records)

n_rdkit2d_fail = int(rdkit2d_df.isna().all(axis=1).sum())
print('rows with a full parse failure (every descriptor NaN):', n_rdkit2d_fail, 'of', len(rdkit2d_df))
assert n_rdkit2d_fail == 0, 'unexpected parse failure on already-canonicalized SMILES -- stopping'

print('rdkit_2d_descriptors produced', rdkit2d_df.shape[1], 'columns for', len(rdkit2d_df), 'compounds')

rdkit version (determines the rdkit_2d_descriptors column set): 2025.09.3


rows with a full parse failure (every descriptor NaN): 0 of 5655
rdkit_2d_descriptors produced 217 columns for 5655 compounds


In [11]:
all_nan_cols = rdkit2d_df.columns[rdkit2d_df.isna().all(axis=0)].tolist()
partial_nan_cols = rdkit2d_df.columns[rdkit2d_df.isna().any(axis=0) & ~rdkit2d_df.isna().all(axis=0)].tolist()
non_nan_df = rdkit2d_df.drop(columns=all_nan_cols)
constant_cols = non_nan_df.columns[non_nan_df.nunique(dropna=True) <= 1].tolist()

print('columns entirely NaN across all', len(rdkit2d_df), 'compounds:', len(all_nan_cols))
print(' ', all_nan_cols)
print('columns with some (but not all) NaN:', len(partial_nan_cols))
print(' ', partial_nan_cols)
print('columns constant (a single value) across all', len(rdkit2d_df), 'compounds:', len(constant_cols))
print(' ', constant_cols)

columns entirely NaN across all 5655 compounds: 0
  []
columns with some (but not all) NaN: 0
  []
columns constant (a single value) across all 5655 compounds: 16
  ['NumRadicalElectrons', 'SMR_VSA8', 'SlogP_VSA9', 'fr_C_S', 'fr_azide', 'fr_azo', 'fr_barbitur', 'fr_diazo', 'fr_epoxide', 'fr_isocyan', 'fr_isothiocyan', 'fr_nitroso', 'fr_phos_acid', 'fr_phos_ester', 'fr_prisulfonamd', 'fr_thiocyan']


**Finding, flagged and not acted on:** the columns printed above (if any) are exactly
what they are in this specific RDKit version/dataset combination -- neither dropped nor
filtered here, per the task brief. Whether to exclude constant or NaN-heavy columns
before model fitting is a feature-selection decision for notebook 04's modelling step,
not this one; this notebook's job is only to freeze and report what the descriptor
calculator actually produces.

In [12]:
tabular_rdkit2d_path = f'{PROCESSED}/tabular_baseline_features_rdkit2d.csv'

tabular_features_rdkit2d = pd.concat(
    [compound_index[['Molecule_Name', 'inchikey', 'split']].reset_index(drop=True), rdkit2d_df],
    axis=1,
)

print('rows before save:', len(tabular_features_rdkit2d))
tabular_features_rdkit2d.to_csv(tabular_rdkit2d_path, index=False)
print('wrote', tabular_rdkit2d_path)

reread_rdkit2d = pd.read_csv(tabular_rdkit2d_path)
print('rows after reading back from disk:', len(reread_rdkit2d))
assert len(reread_rdkit2d) == len(tabular_features_rdkit2d), 'rdkit2d feature file row count changed on round-trip'
assert reread_rdkit2d['inchikey'].is_unique
print('split counts:', reread_rdkit2d['split'].value_counts().to_dict())

rows before save: 5655


wrote data/processed/tabular_baseline_features_rdkit2d.csv
rows after reading back from disk: 5655
split counts: {'train': 4905, 'test': 750}


## 3. CheMeleon embeddings

A single frozen forward pass through the pretrained CheMeleon foundation-model encoder
(chemprop `BondMessagePassing`, mean-pooled via `MeanAggregation` — Zenodo record
15460715) — the checkpoint's weights are loaded and never updated, so this is feature
extraction, not a training run, matching the task brief's framing. Implemented as
`chemeleon_embeddings()` in `src/features.py`, not inline here, for the same
shared-module reason as Section 2.

**Checkpoint:** found already cached at `~/.chemprop/chemeleon_mp.pt` (34.8 MB, chemprop
d_h=2048, depth=6 — this is chemprop's own default foundation-model cache location, so
`chemeleon_embeddings()` uses it with no download step). Output is a 2048-dim float32
vector per compound.

**A real reliability problem found and fixed during development, documented here rather
than silently worked around:** rdkit and torch each bundle their own OpenMP runtime.
Running the encoder's forward pass multi-threaded on this machine segfaults the whole
Python process — reproduced directly and repeatedly while building this function: at
full threading (11 threads) it crashes immediately; at 2 and 4 threads the computation
itself completes and prints a correct result, but the process then segfaults on
interpreter exit; only `OMP_NUM_THREADS=1` (set at the very top of this notebook, before
rdkit/torch/chemprop are first imported anywhere in the process) was reliable across
every trial. MPS (Apple Silicon GPU) was also tried and ruled out on a correctness
basis, not a stability one: chemprop's message-passing aggregation step
(`scatter_reduce`) isn't implemented for the MPS backend in this torch version
(confirmed via a direct `NotImplementedError` from `torch`, not assumed).

**Practical consequence: this cell is genuinely slow (observed ~5-6 minutes for all
5,655 compounds), not "fast" like the rest of this notebook.** Single-threaded CPU is
the only configuration found to be both correct and crash-free, and there's no GPU path
available for this op. Accepted here since it's a one-time frozen-artifact computation
run once, not something re-run per model/fold in 03b -- consistent with "no background
scripts or checkpointing needed" (no *infrastructure* is needed to survive a rerun; it
just isn't instant). `device='cpu'` is also `chemeleon_embeddings()`'s default anyway,
for exact run-to-run determinism on a frozen artifact — the threading finding reinforces
that default rather than overriding it for speed.

**Row order:** `chemeleon_embeddings()` disables dataloader shuffling and forces
`drop_last=False` internally; verified directly during development (not just asserted)
that a batch_size=2 run and a batch_size=1 run over the same 5 compounds produce
bit-identical output in the same order (max abs diff 0.0).

In [13]:
import time

t0 = time.time()
chemeleon_arr = chemeleon_embeddings(smiles_list)
elapsed = time.time() - t0

print('chemeleon_arr shape:', chemeleon_arr.shape, 'dtype:', chemeleon_arr.dtype)
print('elapsed:', round(elapsed, 1), 'seconds for', len(smiles_list), 'compounds')
n_nan = int(np.isnan(chemeleon_arr).sum())
print('NaN values in output:', n_nan)
assert n_nan == 0, 'unexpected NaN in CheMeleon embeddings -- stopping'

chemeleon_arr shape: (5655, 2048) dtype: float32
elapsed: 333.9 seconds for 5655 compounds
NaN values in output: 0


In [14]:
os.makedirs(PROCESSED, exist_ok=True)
embeddings_path = f'{PROCESSED}/chemeleon_embeddings.npy'
embeddings_index_path = f'{PROCESSED}/chemeleon_embeddings_index.csv'

print('rows before save:', len(chemeleon_arr))
np.save(embeddings_path, chemeleon_arr)
compound_index[['Molecule_Name', 'inchikey', 'split']].to_csv(embeddings_index_path, index=False)
print('wrote', embeddings_path)
print('wrote', embeddings_index_path)

reread_embeddings = np.load(embeddings_path)
reread_index = pd.read_csv(embeddings_index_path)
print('rows after reading back from disk:', reread_embeddings.shape[0], '/', len(reread_index))
assert reread_embeddings.shape == chemeleon_arr.shape
assert len(reread_index) == len(compound_index)
assert (reread_index['inchikey'].values == compound_index['inchikey'].values).all(), \
    'embedding index row order does not match compound_index -- alignment broken'
print('alignment confirmed: chemeleon_embeddings.npy row i corresponds to chemeleon_embeddings_index.csv row i.')

rows before save: 5655
wrote data/processed/chemeleon_embeddings.npy
wrote data/processed/chemeleon_embeddings_index.csv
rows after reading back from disk: 5655 / 5655
alignment confirmed: chemeleon_embeddings.npy row i corresponds to chemeleon_embeddings_index.csv row i.


## 4. Multitask target layout

One row per training compound, four target columns (`CYP1A2_pIC50`, `CYP2C9_pIC50`,
`CYP2D6_pIC50`, `CYP3A4_pIC50`), values copied unchanged from the curated
`*_pIC50_direct_inhibition` columns. **NaN handling:** NaN means the compound was not
measured against that isoform in the primary-screen dose-response assay — it is left as
NaN, not imputed, not dropped, and not treated as "inactive" (a measured, non-inhibiting
compound should have a low-but-real pIC50, which is a different thing entirely from
"never tested"). Downstream multitask training must mask NaN targets out of the loss
per-task, not silently zero-fill them.

In [15]:
multitask_targets = train[['Molecule_Name', 'inchikey']].copy()
for iso in ISOFORMS:
    multitask_targets[f'{iso}_pIC50'] = train[PIC50_COLS[iso]].values

print('rows before save:', len(multitask_targets))
print()
print('per-isoform non-null counts (final sanity check against notebook 00):')
expected_non_null = {'CYP1A2': 1412, 'CYP2C9': 1285, 'CYP2D6': 1493, 'CYP3A4': 2335}
all_match = True
for iso in ISOFORMS:
    actual = int(multitask_targets[f'{iso}_pIC50'].notna().sum())
    expected = expected_non_null[iso]
    match = actual == expected
    all_match = all_match and match
    print(f'  {iso}: expected {expected}, got {actual} -- {"MATCH" if match else "MISMATCH"}')
assert all_match, 'multitask target non-null counts do not match notebook 00 -- stopping'

n_all_null = int(multitask_targets[[f'{iso}_pIC50' for iso in ISOFORMS]].isna().all(axis=1).sum())
print()
print('compounds with all four isoforms NaN (should be 0 -- every training compound has >=1 label):', n_all_null)
assert n_all_null == 0

rows before save: 4905

per-isoform non-null counts (final sanity check against notebook 00):
  CYP1A2: expected 1412, got 1412 -- MATCH
  CYP2C9: expected 1285, got 1285 -- MATCH
  CYP2D6: expected 1493, got 1493 -- MATCH
  CYP3A4: expected 2335, got 2335 -- MATCH

compounds with all four isoforms NaN (should be 0 -- every training compound has >=1 label): 0


In [16]:
targets_path = f'{PROCESSED}/multitask_targets.csv'
multitask_targets.to_csv(targets_path, index=False)
print('wrote', targets_path)

reread_targets = pd.read_csv(targets_path)
print('rows after reading back from disk:', len(reread_targets))
assert len(reread_targets) == len(multitask_targets)

wrote data/processed/multitask_targets.csv
rows after reading back from disk: 4905


## 5. Single-concentration primary screen — hit-calling and compound-count reconciliation

Resolves the mismatch flagged as open since `00_schema_audit.ipynb`, now checked
against the challenge documentation's actual hit-calling mechanism rather than a flat
compound-list comparison. The documentation describes the single-concentration primary
screen (`cyp-challenge-single-concentration-TRAIN.csv`) as calling a compound a **hit**
when **`log2fc_estimate < -1` AND `log2fc_fdr < 0.05`**, and states that ~1,500 hit
compounds were prioritized for dose-response-curve (DRC) follow-up — the pIC50 assay
that became `TRAIN_inhibition.csv`. This number is a hard prerequisite for 03b's
design (specifically, whether log2fc pretraining can be restricted to compounds with no
pIC50 label at all, to avoid any leakage concern into the CV comparison) — reported
here as input to that upcoming decision; 03b's actual logic is not built here.

This section (a) loads the raw file directly and confirms its structure before
computing anything, (b) applies the documented hit formula and checks the result
against the documented ~1,500 figure — without forcing a match if it doesn't hold, (c)
cross-references by InChIKey (not `Molecule_Name`, per notebook 01's identity
convention) to break the 4,905 training compounds into hit / measured-not-hit /
no-record buckets, and (d) states plainly what this means for 03b's leakage-avoidance
design.

### 5a. Load the raw file and confirm its structure

Loaded directly from `data/raw/` (not the curated output) since hit-calling needs the
original `log2fc_estimate`/`log2fc_fdr` columns, which aren't carried into any curated
file. Checked before computing anything: is this long-format with multiple rows per
compound (per isoform, per replicate), and if so, how should the hit formula be applied
-- per compound-isoform pair, or aggregated first?

In [17]:
sc_raw = pd.read_csv(f'{RAW}/cyp-challenge-single-concentration-TRAIN.csv')
print('single-concentration file shape:', sc_raw.shape)
print('columns:', list(sc_raw.columns))
print()
print('null count per column:')
print(sc_raw.isna().sum().to_string())
print()
print('rows per enzyme:')
print(sc_raw['enzyme'].value_counts().to_string())
print()
rows_per_compound = sc_raw.groupby('Molecule_Name').size()
rows_per_compound_enzyme = sc_raw.groupby(['Molecule_Name', 'enzyme']).size()
print('rows per compound -- distinct values found:', sorted(rows_per_compound.unique().tolist()))
print('rows per (compound, enzyme) pair -- distinct values found:', sorted(rows_per_compound_enzyme.unique().tolist()))

single-concentration file shape: (17504, 12)
columns: ['Molecule_Name', 'SMILES', 'OCNT_Batch', 'enzyme', 'plate_id', 'concentration_M', 'log2fc_estimate', 'log2fc_std_error', 'p_value', 'log2fc_fdr', 'log2fc_median', 'cohens_d']

null count per column:
Molecule_Name       0
SMILES              0
OCNT_Batch          0
enzyme              0
plate_id            0
concentration_M     0
log2fc_estimate     0
log2fc_std_error    0
p_value             0
log2fc_fdr          0
log2fc_median       0
cohens_d            0

rows per enzyme:
enzyme
CYP1A2    4376
CYP2C9    4376
CYP2D6    4376
CYP3A4    4376

rows per compound -- distinct values found: [4]
rows per (compound, enzyme) pair -- distinct values found: [1]


**Structure confirmed:** 17,504 rows = 4,376 unique compounds × exactly 4 rows each
(one per isoform: CYP1A2/CYP2C9/CYP2D6/CYP3A4), and exactly 1 row per (compound,
isoform) pair -- no replicate rows to aggregate, and no partial panels (every screened
compound has all four isoforms present). Zero nulls anywhere.

**Interpretation used below, stated before computing anything:** each row already
carries its own independently-computed `log2fc_estimate`/`log2fc_fdr` for that one
isoform, so the documented formula is unambiguous at the (compound, isoform) row level
-- there's no aggregation choice to make there. The aggregation question is only how to
roll a per-isoform hit flag up to a compound-level "hit" label, to compare against the
documented "~1,500 hit compounds." Used here: a compound counts as a hit if it clears
the threshold on **at least one** of its four isoforms (ANY-isoform aggregation) -- this
matches how a primary-screen triage funnel works in practice (a compound gets flagged
for DRC follow-up if it shows a significant inhibitory signal against any one panel, not
only if it hits all four simultaneously), and it's also the aggregation level the pIC50
training set itself uses (a compound is "in" that set if it has a label for even one
isoform).

### 5b. Apply the documented hit-calling formula

`log2fc_estimate < -1` AND `log2fc_fdr < 0.05`, applied exactly as documented, at the
(compound, isoform) row level per 5a. Rolled up to a compound-level hit flag via
ANY-isoform-hit, then checked directly against the documented ~1,500 figure.

In [18]:
sc_raw['is_hit_row'] = (sc_raw['log2fc_estimate'] < -1) & (sc_raw['log2fc_fdr'] < 0.05)
n_hit_rows = int(sc_raw['is_hit_row'].sum())
print('(compound, isoform) rows called a hit:', n_hit_rows, 'of', len(sc_raw))
print()
print('hit rows per isoform:')
print(sc_raw.groupby('enzyme')['is_hit_row'].sum().to_string())
print()

compound_hits = sc_raw.groupby('Molecule_Name').agg(
    SMILES=('SMILES', 'first'),
    n_isoforms_hit=('is_hit_row', 'sum'),
).reset_index()
compound_hits['is_hit_compound'] = compound_hits['n_isoforms_hit'] > 0

n_hit_compounds = int(compound_hits['is_hit_compound'].sum())
print('distinct compounds hit on >=1 isoform:', n_hit_compounds, 'of', len(compound_hits),
      f'({n_hit_compounds / len(compound_hits) * 100:.1f}%)')
print()
print('distribution of # isoforms hit, among compounds with >=1 hit:')
print(compound_hits.loc[compound_hits['is_hit_compound'], 'n_isoforms_hit'].value_counts().sort_index().to_string())
print()
print('documented figure: ~1,500 hit compounds prioritized for DRC follow-up')
print(f'computed figure (ANY-isoform hit): {n_hit_compounds}')
print(f'ratio to documented figure: {n_hit_compounds / 1500:.2f}x')

(compound, isoform) rows called a hit: 6868 of 17504

hit rows per isoform:
enzyme
CYP1A2    1307
CYP2C9    1452
CYP2D6    1198
CYP3A4    2911

distinct compounds hit on >=1 isoform: 3326 of 4376 (76.0%)

distribution of # isoforms hit, among compounds with >=1 hit:
n_isoforms_hit
1    1186
2    1060
3     758
4     322

documented figure: ~1,500 hit compounds prioritized for DRC follow-up
computed figure (ANY-isoform hit): 3326
ratio to documented figure: 2.22x


**Finding: the computed hit count does not match the documented ~1,500 figure, and
this is not forced to reconcile.** ANY-isoform hit-calling on the documented formula
gives 3,326 compounds -- more than double the documented ~1,500. No single reasonable
aggregation of the per-isoform counts lands close either: CYP1A2 1,307 / CYP2C9 1,452 /
CYP2D6 1,198 / CYP3A4 2,911 hit rows per isoform (none of the four alone matches, though
CYP2C9's 1,452 is the closest single value found, ~3.2% below 1,500 -- noted as a
pattern, not confirmed as the explanation). "Hit on exactly 1 isoform" (1,186) is also
in the right neighborhood but not a match either.

**Plausible explanations, not adjudicated here:**
1. **"Prioritized for DRC follow-up" likely involved criteria beyond the two documented
   statistical thresholds.** The raw file carries `p_value`, `log2fc_median`, and
   `cohens_d` columns that play no role in the documented formula -- real-world
   prioritization (capacity-constrained, since DRC follow-up is expensive) plausibly
   used additional filtering (effect-size ranking, redundancy/analog deduplication,
   counterscreen artifact removal) on top of the two documented thresholds.
2. **"Hit" (statistically significant) and "prioritized" (selected for follow-up) may
   simply be two different sets by design** -- not every statistical hit needs its own
   DRC run if it's structurally redundant with an already-prioritized compound.
3. **The ~1,500 figure may describe a single-isoform screen, not a cross-isoform
   compound-level total** -- CYP2C9's 1,452 hit rows is the closest match found to
   ~1,500 of anything computed here, though there's no documentation tying the figure
   specifically to CYP2C9 rather than a cross-isoform count, so this is speculative.

Not resolved further here -- flagged for follow-up with OpenADMET directly if it
matters to a later modelling decision, per the same pattern as the schema audit's
unresolved `_conf_high`/`_conf_low` question.

### 5c. Cross-reference by InChIKey against the pIC50 training set

Matched by InChIKey (not `Molecule_Name`) -- the identity check established in notebook
01, since `Molecule_Name` can differ across batches/registrations of the same compound
while InChIKey reflects the actual structure. `add_canonical_smiles_and_inchikey` from
`src/features.py` is reused here, not reimplemented.

In [19]:
from src.features import add_canonical_smiles_and_inchikey

compound_hits_c = add_canonical_smiles_and_inchikey(compound_hits, smiles_col='SMILES')
n_parse_fail = int(compound_hits_c['inchikey'].isna().sum())
print('SMILES parse failures:', n_parse_fail, 'of', len(compound_hits_c))
assert n_parse_fail == 0, 'unexpected parse failure -- stopping'

dup_ik_mask = compound_hits_c['inchikey'].duplicated(keep=False)
n_dup_ik_rows = int(dup_ik_mask.sum())
print('Molecule_Name rows sharing a duplicate InChIKey within this file:', n_dup_ik_rows)
if n_dup_ik_rows:
    display(
        compound_hits_c.loc[dup_ik_mask, ['Molecule_Name', 'inchikey', 'is_hit_compound']]
        .sort_values('inchikey')
    )

n_unique_by_name = compound_hits_c['Molecule_Name'].nunique()
n_unique_by_ik = compound_hits_c['inchikey'].nunique()
print()
print('unique compounds by Molecule_Name:', n_unique_by_name)
print('unique compounds by InChIKey:     ', n_unique_by_ik)

SMILES parse failures: 0 of 4376
Molecule_Name rows sharing a duplicate InChIKey within this file: 2


,Molecule_Name,inchikey,is_hit_compound
4219,OCNT-2328450,QMHSXPLYMTVAMK-UHFFFAOYSA-N,True
4271,OCNT-2328658,QMHSXPLYMTVAMK-UHFFFAOYSA-N,True



unique compounds by Molecule_Name: 4376
unique compounds by InChIKey:      4375


In [20]:
sc_ik_hit = set(compound_hits_c.loc[compound_hits_c['is_hit_compound'], 'inchikey'])
sc_ik_tested = set(compound_hits_c['inchikey'])
train_ik = set(train['inchikey'])

bucket_hit = train_ik & sc_ik_hit
bucket_measured_not_hit = (train_ik & sc_ik_tested) - sc_ik_hit
bucket_no_record = train_ik - sc_ik_tested

assert len(bucket_hit) + len(bucket_measured_not_hit) + len(bucket_no_record) == len(train_ik), \
    'bucket counts do not sum to the full training set -- stopping'

print('three-bucket breakdown of the 4,905 pIC50-labeled training compounds, by InChIKey:')
for name, b in [('called a hit in the primary screen', bucket_hit),
                ('measured in the primary screen, NOT a hit', bucket_measured_not_hit),
                ('no primary-screen record at all', bucket_no_record)]:
    print(f'  {name}: {len(b)} ({len(b) / len(train_ik) * 100:.1f}%)')

sc_ik_only = sc_ik_tested - train_ik
print()
print('single-concentration compounds (by InChIKey) with NO pIC50 label at all:',
      len(sc_ik_only), 'of', n_unique_by_ik)

three-bucket breakdown of the 4,905 pIC50-labeled training compounds, by InChIKey:
  called a hit in the primary screen: 3325 (67.8%)
  measured in the primary screen, NOT a hit: 1050 (21.4%)
  no primary-screen record at all: 530 (10.8%)

single-concentration compounds (by InChIKey) with NO pIC50 label at all: 0 of 4375


**Finding: the three-bucket breakdown.** Of the 4,905 pIC50-labeled training compounds:
**3,325 (67.8%)** were called a hit in the primary screen, **1,050 (21.4%)** were
measured but not called a hit, and **530 (10.8%)** have no primary-screen record at
all. These three buckets sum exactly to 4,905, confirmed.

**A correction to the earlier Molecule_Name-based finding, and exactly why InChIKey
matching matters here:** the previous version of this section (matching on
`Molecule_Name`) found 1 single-concentration compound with no pIC50 label at all.
Matching by InChIKey instead finds **0** -- every one of the 4,375 unique compounds (by
InChIKey; 4,376 unique `Molecule_Name`s collapse to 4,375 once the one duplicate
InChIKey pair above is resolved) in the single-concentration file has a pIC50 label
somewhere in the training set. The single Molecule_Name-only compound from before
(`OCNT-2328658`, single-concentration file only by name) turns out to be chemically
identical (same InChIKey) to `OCNT-2328450`, which *is* in the pIC50 training set under
a different `Molecule_Name` -- two different batch/registration IDs for the same
structure, exactly the scenario notebook 01 established InChIKey matching to catch.
Both are independently called hits above, so this correction doesn't affect the hit
count, only the "no pIC50 label" count -- which drops from 1 to 0.

## 6. Final summary

**Artifacts generated (all frozen — treat as immutable from this point forward):**
- `data/folds/cv_folds.csv` — 4,905 training compounds × (`Molecule_Name`, `inchikey`,
  `repeat_0`..`repeat_4`), plain 5×5 repeated random CV, `SEED = 42`.
- `data/processed/tabular_baseline_features.csv` — 5,655 compounds (train ∪ test) ×
  (`Molecule_Name`, `inchikey`, `split`, 2048 ECFP4 bits, 9 physicochemical
  descriptors) -- the narrow, SAR-motivated descriptor option (Section 2a).
- `data/processed/tabular_baseline_features_rdkit2d.csv` — the same 5,655 compounds ×
  (`Molecule_Name`, `inchikey`, `split`, full RDKit 2D descriptor set) -- the broader
  option matching the official baseline models' feature recipe (Section 2b); saved
  separately from, not merged into, the file above.
- `data/processed/chemeleon_embeddings.npy` + `chemeleon_embeddings_index.csv` — 5,655
  × 2048 float32 CheMeleon embeddings, with a matching-row-order compound index.
- `data/processed/multitask_targets.csv` — 4,905 training compounds ×
  (`Molecule_Name`, `inchikey`, 4 isoform pIC50 columns, NaN = not measured against
  that isoform).
- `data/processed/log2fc_targets.csv` — 4,376 single-concentration-screened compounds ×
  (`Molecule_Name`, `inchikey`, 4 isoform log2fc columns) — the log2fc analogue of
  `multitask_targets.csv` (Section 7). No NaN handling needed (every screened compound
  has all four isoforms, confirmed in Section 5a); one row per `Molecule_Name`, not
  deduplicated by `inchikey` -- the one known duplicate-InChIKey pair carries genuinely
  different measured values and is kept as two rows rather than merged.

**Decisions made and reasoned through here (not silently picked):**
1. **CYP2D6 CV stratification** — not applied. No assay-protocol distinction is
   documented in the released files, and per-fold CYP2D6 label representation is
   already balanced (4.3% relative std, fold counts within ~8% of the expected 298.6)
   under plain random splitting, checked directly in Section 1a. All four isoforms
   treated uniformly for CV purposes.
2. **Fold-generation seed handling** — one master seed (`SEED = 42`) passed to
   `RepeatedKFold`; verified (both from `sklearn`'s source and empirically, Section 1b)
   that this produces 5 genuinely distinct partitions, not 5 copies of one split.
   `CLAUDE.md`'s per-fold seed-variance rule is about model-fitting seeds, which belong
   to 03b — no model is fit here.
3. **Reusing `isoform_structural_descriptors`** (built in notebook 02 for isoform SAR
   checks) as the physicochemical descriptor set here — it's the only such function
   that exists in `src/features.py`, so reusing it satisfies the "don't reimplement"
   instruction rather than violating it.
4. **CheMeleon threading/device** — `OMP_NUM_THREADS=1`, CPU-only. Not a preference;
   multi-threaded CPU segfaults (reproduced directly) and MPS doesn't implement the op
   chemprop's aggregation step needs (confirmed via `torch`'s own error, not assumed).
   Documented fully in Section 3, including the real runtime cost (~5-6 min for that
   one cell).
5. **Adding `rdkit_2d_descriptors` as a second, broader descriptor option (Section 2b)**
   — not a replacement for `isoform_structural_descriptors`, which stays as the narrow,
   SAR-motivated option for the purpose notebook 02 built it for. Added because the
   official baseline models (XGB-baseline/LGBM-baseline) use the full RDKit 2D set +
   ECFP4, and benchmarking against them needs that same feature recipe available.
   Constant/all-NaN columns found in the output (Section 2b) are flagged, not dropped
   -- feature selection is notebook 04's decision, not this one.

**What was found:**
- Hit-calling (Section 5b): the documented formula (`log2fc_estimate < -1` AND
  `log2fc_fdr < 0.05`) gives 3,326 ANY-isoform-hit compounds (Molecule_Name-level) --
  more than double the documented "~1,500 prioritized for DRC follow-up," and no
  single reasonable aggregation of the per-isoform counts (CYP1A2 1,307 / CYP2C9 1,452
  / CYP2D6 1,198 / CYP3A4 2,911) matches either. Not forced to reconcile; three
  plausible explanations are listed in Section 5b without picking one.
- Three-bucket breakdown (Section 5c, by InChIKey): of the 4,905 pIC50-labeled training
  compounds, 3,325 (67.8%) were a primary-screen hit, 1,050 (21.4%) were measured but
  not a hit, and 530 (10.8%) have no primary-screen record at all.
- **The InChIKey-based re-check overturns the earlier Molecule_Name-based finding**:
  0 (not 1) single-concentration compounds have no pIC50 label at all -- the one
  apparent exception was a duplicate-batch-ID case (`OCNT-2328658`/`OCNT-2328450`,
  same InChIKey). Given essentially 100% of the single-concentration population already
  carries a pIC50 label, and hit-status doesn't change that (hit and non-hit compounds
  are both drawn almost entirely from the same already-labeled population), **"restrict
  log2fc pretraining to compounds with no pIC50 label at all" is not a viable
  leakage-avoidance strategy for 03b** -- there is effectively no unlabeled pool to
  restrict to. 03b will need one of the other approaches already under discussion:
  per-fold pretraining (re-pretraining per CV fold on only that fold's training
  compounds) or accepting the near-total overlap as a documented, CheMeleon-precedented
  assumption (large pretrained encoders are routinely trained on data that overlaps
  downstream tasks; CheMeleon itself is an instance of exactly this).
- No compound-count or alignment issues between any generated artifact and the
  `01_data_curation.ipynb` curated data — checked explicitly (not assumed) in Sections
  1c, 2, 3, and 4.

**What remains open for 03b (and beyond) to pick up:**
- Which leakage-avoidance strategy to actually use for log2fc pretraining, given
  Section 5's finding that a label-based exclusion filter isn't viable (per-fold
  pretraining vs. accepting the overlap as a documented assumption).
- The unresolved ~1,500-vs-3,326 hit-count discrepancy from Section 5b, if it becomes
  relevant to a later modelling decision (e.g. if 03b wants to use hit-status as a
  pretraining signal or sample weight).
- The chemistry-motivated / cluster-based split, explicitly out of scope in this
  notebook per the task brief.
- Whether to drop or keep the constant/all-NaN columns flagged in Section 2b's RDKit 2D
  descriptor output -- a feature-selection call for notebook 04, not decided here.
- Any actual model training or CV evaluation — this notebook only freezes inputs.

## 7. Log2fc target table — multitask layout for the primary screen

The log2fc equivalent of Section 4's `multitask_targets.csv`: one row per
single-concentration-screened compound, four columns (`CYP1A2_log2fc`,
`CYP2C9_log2fc`, `CYP2D6_log2fc`, `CYP3A4_log2fc`), reshaped directly from the
long-format `sc_raw` table already loaded in Section 5 -- no reload, no
recomputation, values taken as-is from `log2fc_estimate`.

**No missingness decision needed here, unlike Section 4:** Section 5a already
established that every screened compound has exactly one row per isoform, all four
isoforms present, zero nulls -- so there is no NaN-handling logic to add. Confirmed
directly below (asserted, not assumed) rather than adding handling for a case that
can't occur.

In [21]:
log2fc_wide = sc_raw.pivot(index='Molecule_Name', columns='enzyme', values='log2fc_estimate')
log2fc_wide.columns.name = None
log2fc_wide = log2fc_wide.rename(columns={iso: f'{iso}_log2fc' for iso in ISOFORMS}).reset_index()

log2fc_cols = [f'{iso}_log2fc' for iso in ISOFORMS]
log2fc_targets = log2fc_wide.merge(
    compound_hits_c[['Molecule_Name', 'inchikey']], on='Molecule_Name', how='left'
)[['Molecule_Name', 'inchikey'] + log2fc_cols]

print('rows (one per single-concentration-screened Molecule_Name):', len(log2fc_targets))

n_nan = int(log2fc_targets[log2fc_cols].isna().sum().sum())
print('NaN values across all four log2fc columns:', n_nan)
assert n_nan == 0, (
    'unexpected NaN -- Section 5a found zero nulls and a full 4-isoform panel per '
    'compound, so this should never trigger'
)

rows (one per single-concentration-screened Molecule_Name): 4376
NaN values across all four log2fc columns: 0


In [22]:
n_unique_by_name = log2fc_targets['Molecule_Name'].nunique()
n_unique_by_ik = log2fc_targets['inchikey'].nunique()
print('unique compounds by Molecule_Name:', n_unique_by_name, '(matches Section 5a\'s 4,376 finding)')
print('unique compounds by InChIKey:     ', n_unique_by_ik, '(matches Section 5c\'s InChIKey-deduplicated count)')

if n_unique_by_name != n_unique_by_ik:
    dup_names_here = compound_hits_c.loc[dup_ik_mask, 'Molecule_Name'].tolist()
    print()
    print(f'Row count does not collapse to the InChIKey count -- differ by '
          f'{n_unique_by_name - n_unique_by_ik}, exactly the duplicate-InChIKey pair '
          f'already found in Section 5c: {dup_names_here}.')
    print('Their actual log2fc_estimate values, shown directly rather than assumed identical:')
    display(log2fc_targets[log2fc_targets['Molecule_Name'].isin(dup_names_here)])

unique compounds by Molecule_Name: 4376 (matches Section 5a's 4,376 finding)
unique compounds by InChIKey:      4375 (matches Section 5c's InChIKey-deduplicated count)

Row count does not collapse to the InChIKey count -- differ by 1, exactly the duplicate-InChIKey pair already found in Section 5c: ['OCNT-2328450', 'OCNT-2328658'].
Their actual log2fc_estimate values, shown directly rather than assumed identical:


,Molecule_Name,inchikey,CYP1A2_log2fc,CYP2C9_log2fc,CYP2D6_log2fc,CYP3A4_log2fc
4219,OCNT-2328450,QMHSXPLYMTVAMK-UHFFFAOYSA-N,-0.894689,-1.051808,-2.392894,-2.945988
4271,OCNT-2328658,QMHSXPLYMTVAMK-UHFFFAOYSA-N,-0.535052,-0.858630,-0.665606,-1.877521


**Finding: the two Molecule_Names sharing one InChIKey carry genuinely different
log2fc values per isoform** (different batches/registrations of the same compound,
independently screened) -- not near-duplicates that happen to round differently, but
substantively different measurements across all four isoforms. Averaging or picking
one would be a modelling decision this notebook doesn't make unprompted, so **this
artifact is kept at one row per `Molecule_Name` (4,376 rows), not collapsed to one row
per InChIKey (4,375)** -- both rows for the duplicate pair are retained, `inchikey` is
carried as a column (not the index) precisely so this is visible and actionable rather
than silently resolved. This mirrors Section 4's `multitask_targets.csv`, which is
also one row per `Molecule_Name` with `inchikey` as a plain column, not deduplicated by
it either.

In [23]:
log2fc_targets_path = f'{PROCESSED}/log2fc_targets.csv'

print('rows before save:', len(log2fc_targets))
log2fc_targets.to_csv(log2fc_targets_path, index=False)
print('wrote', log2fc_targets_path)

reread_log2fc = pd.read_csv(log2fc_targets_path)
print('rows after reading back from disk:', len(reread_log2fc))
assert len(reread_log2fc) == len(log2fc_targets), 'log2fc target file row count changed on round-trip'
assert reread_log2fc[log2fc_cols].isna().sum().sum() == 0

rows before save: 4376
wrote data/processed/log2fc_targets.csv
rows after reading back from disk: 4376


## Before finishing, check against CLAUDE.md

- **Row counts logged before/after every step:** yes — Section 0 (load), Section 1c
  (fold file save + reread), Section 2 (feature file save + reread), Section 3
  (embeddings save + reread), and Section 4 (targets save + reread) all print row
  counts before and after.
- **Every seed used is logged:** yes — `SEED = 42` is printed in the setup cell before
  any use, and is the only source of randomness anywhere in this notebook (no model
  fitting happens here, so no per-fold model-training seeds apply).
- **Fingerprints/descriptors/splits go through one shared module
  (`src/features.py`):** yes — `ecfp4_fingerprints`, `isoform_structural_descriptors`,
  and the new `chemeleon_embeddings` are all defined there and imported in the setup
  cell, not redefined inline. `RepeatedKFold` (the CV split mechanism itself) is used
  directly from `sklearn`, not wrapped, since it's a single call with no
  project-specific logic to centralize.
- **Every modelling-adjacent decision not explicitly settled by the task brief flagged
  and reasoned through, not silently picked:** four such decisions are documented
  inline and summarized in Section 6 above (CYP2D6 stratification, fold-seed handling,
  descriptor-function reuse, CheMeleon threading/device).
- **No filtering/dropping without a logged before/after count:** confirmed — this
  notebook never drops a row; every assertion added is a stop-on-mismatch check, not a
  filter.
- **Nothing here touches test-set labels:** confirmed — the test set has no label
  columns to touch (per `01_data_curation.ipynb`); only its `canonical_smiles` is used,
  for feature/embedding generation, which is not a modelling decision that could leak
  test-set information into training.